# Scrollie — Dafne Water Segmentation Viewer

- **Left**: Dixon WATER image
- **Right**: Dafne thigh segmentation overlay

In [1]:
import glob
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as _cm
import SimpleITK as sitk
from ipywidgets import IntSlider, Dropdown, VBox
import ipywidgets as widgets
from IPython.display import display

In [2]:
SEG_DIR   = os.path.join('..', 'dafne_water_segs')
DATA_ROOT = os.path.join('..', 'myosegmenTUM')

npz_files = sorted(glob.glob(os.path.join(SEG_DIR, '*_dafne_thigh.npz')))
file_options = {
    os.path.basename(p).replace('_dafne_thigh.npz', ''): p
    for p in npz_files
}

def npz_to_nii(stem):
    # stem e.g. HV001_1_WATER_stack1
    m = re.match(r'(.+)_WATER_(stack\d+)$', stem)
    subject = m.group(1)
    stack   = m.group(2)
    return os.path.join(DATA_ROOT, subject, 'ImageData',
                        f'{subject}_WATER',
                        f'{subject}_WATER_{stack}.nii')

print(f'Found {len(file_options)} segmented stacks')
if file_options:
    first = list(file_options)[0]
    print(f'Image path example: {npz_to_nii(first)}')

Found 46 segmented stacks
Image path example: ..\myosegmenTUM\HV001_1\ImageData\HV001_1_WATER\HV001_1_WATER_stack1.nii


In [3]:
cmap = _cm.get_cmap('tab20', 20)

def build_overlay(npz_data, n_slices, h, w):
    overlay  = np.zeros((n_slices, h, w, 4), dtype=float)
    patches  = []
    for i, name in enumerate(npz_data.files):
        color = cmap(i % 20)
        mask  = npz_data[name]  # (slices, H, W)
        overlay[mask > 0] = [color[0], color[1], color[2], 0.5]
        patches.append(mpatches.Patch(color=color, alpha=0.6, label=name))
    return overlay, patches

def load_stack(label):
    npz_path = file_options[label]
    nii_path = npz_to_nii(label)

    img_arr  = sitk.GetArrayFromImage(sitk.ReadImage(nii_path)).astype(float)
    img_norm = (img_arr - img_arr.min()) / (img_arr.max() - img_arr.min() + 1e-8)

    npz_data         = np.load(npz_path)
    overlay, patches = build_overlay(npz_data, *img_arr.shape)
    return img_norm, overlay, patches

C:\Users\docto\AppData\Local\Temp\ipykernel_33596\708560632.py:1: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = _cm.get_cmap('tab20', 20)


In [4]:
file_dropdown = Dropdown(options=list(file_options.keys()), description='Stack:')
slice_slider  = IntSlider(min=0, max=1, step=1, value=0, description='Slice:',
                          layout=widgets.Layout(width='600px'))
out = widgets.Output()

_cache = {}

def get_data(label):
    if label not in _cache:
        img_norm, overlay, patches = load_stack(label)
        _cache[label] = (img_norm, overlay, patches)
        slice_slider.max   = img_norm.shape[0] - 1
        slice_slider.value = 0
    return _cache[label]

def render(label, slice_idx):
    img_norm, overlay, patches = get_data(label)
    img = img_norm[slice_idx]

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    axes[0].imshow(img, cmap='gray', origin='lower')
    axes[0].set_title(f'WATER image — slice {slice_idx}')
    axes[0].axis('off')

    axes[1].imshow(img, cmap='gray', origin='lower')
    axes[1].imshow(overlay[slice_idx], origin='lower')
    axes[1].set_title('Dafne Thigh Segmentation')
    axes[1].axis('off')
    axes[1].legend(handles=patches, loc='lower right', fontsize=6, framealpha=0.7)

    fig.suptitle(label, fontsize=10)
    plt.tight_layout()
    with out:
        out.clear_output(wait=True)
        plt.show()

def on_file_change(change):
    _cache.clear()
    get_data(change['new'])
    render(file_dropdown.value, slice_slider.value)

def on_slice_change(change):
    render(file_dropdown.value, change['new'])

file_dropdown.observe(on_file_change, names='value')
slice_slider.observe(on_slice_change, names='value')

if file_options:
    get_data(file_dropdown.value)
    render(file_dropdown.value, 0)

display(VBox([file_dropdown, slice_slider, out]))